# Hybrid Retrieval for RAG
### BM25 (Sparse) + BAAI/bge-large-en-v1.5 (Dense) fused with Reciprocal Rank Fusion (RRF)

This notebook replicates the PDF-loading and chunking pipeline from `PDF_chunking_embedding.ipynb`
and adds a **Hybrid Retriever** that merges keyword-based BM25 and semantic dense retrieval
via LangChain's `EnsembleRetriever` (which uses RRF internally).

## Step 0: Install dependencies

In [ ]:
!pip install -q langchain langchain-community langchain-huggingface langchain-text-splitters chromadb rank_bm25 pypdf sentence-transformers

## Step 1: Load the PDF

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "../Data/odisha_judgement_files/display_judgement20pdf.pdf"

loader = PyPDFLoader(file_path)
documents = loader.load()

print(f"Loaded {len(documents)} pages.")

## Step 2: Chunk the document (same settings as PDF_chunking_embedding.ipynb)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=[
        "\n\n",   # Split by major paragraphs first
        "\n",     # Then split by line breaks
        ". ",     # Then split by sentences
        ", ",     # Then split by clauses
        " "       # Last resort: split by words
    ]
)

chunks = text_splitter.split_documents(documents)
print(f"Total chunks created: {len(chunks)}")
print(f"\nSample chunk (chunk 0):\n{chunks[0].page_content[:400]}")

## Step 3: Sparse Retriever — BM25 (Keyword-based)

BM25 ranks documents by **term frequency** and **inverse document frequency**. It excels at finding chunks that contain exact legal keywords (case numbers, section names, party names).

In [ ]:
from langchain_community.retrievers import BM25Retriever

# No embeddings needed — BM25 works directly on raw text
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 5  # Retrieve top-5 candidates from keyword search

print("BM25 retriever initialized.")

# Quick test
test_query = "fresh investigation foul play"
bm25_results = bm25_retriever.invoke(test_query)
print(f"\nBM25 top results for: '{test_query}'")
for i, doc in enumerate(bm25_results):
    print(f"  [{i+1}] Page {doc.metadata.get('page')} | {doc.page_content[:120]}...")

## Step 4: Dense Retriever — BAAI/bge-large-en-v1.5 + Chroma (Semantic)

The `BAAI/bge-large-en-v1.5` encoder converts text to 1024-dim embeddings. It captures **semantic meaning** even when exact keywords differ. `normalize_embeddings=True` is required for correct cosine similarity with BGE models.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en-v1.5",
    encode_kwargs={"normalize_embeddings": True},   # CRITICAL for BGE models
    model_kwargs={"device": "cpu"}                  # Change to 'mps' on Apple Silicon or 'cuda' on GPU
)

print("BGE embedding model loaded.")

In [ ]:
from langchain_community.vectorstores import Chroma

print("Building Chroma vector store from chunks... (this may take a minute)")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="judgement_hybrid_retrieval"
)

dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})  # Retrieve top-5 candidates

print(f"Vector store ready — {vectorstore._collection.count()} chunks indexed.")

# Quick test
# NOTE: BGE recommends adding an instruction prefix for retrieval queries
BGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
test_query = "fresh investigation foul play"
dense_results = dense_retriever.invoke(BGE_QUERY_PREFIX + test_query)
print(f"\nDense top results for: '{test_query}'")
for i, doc in enumerate(dense_results):
    print(f"  [{i+1}] Page {doc.metadata.get('page')} | {doc.page_content[:120]}...")

## Step 5: Hybrid Retrieval with Reciprocal Rank Fusion (RRF)

RRF score formula:
```
RRF_score(doc) = sum(  1 / (k + rank_i(doc))  )   for each retriever i
```
where `k = 60` by default. A document ranking **high in both** lists gets the best final score.

| Weight config | Best for |
|---|---|
| `[0.5, 0.5]` | Balanced queries |
| `[0.6, 0.4]` | Keyword-heavy (section numbers, party names) |
| `[0.4, 0.6]` | Conceptual / paraphrased queries |

In [ ]:
from langchain.retrievers import EnsembleRetriever

# weights=[BM25_weight, Dense_weight]
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.5, 0.5]
)

print("Ensemble retriever (BM25 + BGE Dense with RRF) is ready.")

## Step 6: Run Hybrid Retrieval Queries

In [ ]:
def hybrid_retrieve(query: str, top_k: int = 5):
    """
    Run hybrid retrieval (BM25 + BGE dense) with RRF fusion.
    Returns top_k chunks ranked by fused RRF score.
    """
    results = ensemble_retriever.invoke(query)
    return results[:top_k]


# --- Query 1: Keyword-heavy legal query ---
query1 = "fresh investigation foul play Birendra Lakra"
print("=" * 60)
print(f" QUERY 1: {query1}")
print("=" * 60)
results1 = hybrid_retrieve(query1)
for i, doc in enumerate(results1):
    src = doc.metadata.get('source', 'N/A').split('/')[-1]
    print(f"\n[Rank {i+1}] Page: {doc.metadata.get('page')} | File: {src}")
    print(doc.page_content[:400] + "...")


# --- Query 2: Conceptual / paraphrased query ---
query2 = "What were the court observations about the impartiality of the investigating officer?"
print("\n" + "=" * 60)
print(f" QUERY 2: {query2}")
print("=" * 60)
results2 = hybrid_retrieve(query2)
for i, doc in enumerate(results2):
    src = doc.metadata.get('source', 'N/A').split('/')[-1]
    print(f"\n[Rank {i+1}] Page: {doc.metadata.get('page')} | File: {src}")
    print(doc.page_content[:400] + "...")

## Step 7: Side-by-side Comparison — BM25 vs Dense vs Hybrid

In [ ]:
def compare_retrievers(query: str, top_k: int = 3):
    """Side-by-side comparison of BM25, Dense (BGE), and Hybrid (RRF) retrieval."""
    print(f"\nQuery: {query}\n")

    bm25_res   = bm25_retriever.invoke(query)[:top_k]
    dense_res  = dense_retriever.invoke(query)[:top_k]
    hybrid_res = ensemble_retriever.invoke(query)[:top_k]

    headers = ["BM25 (Keyword)", "Dense — BGE Semantic", "Hybrid RRF Fusion"]
    all_results = [bm25_res, dense_res, hybrid_res]

    for header, res in zip(headers, all_results):
        print("-" * 60)
        print(f"  {header}")
        print("-" * 60)
        for i, doc in enumerate(res):
            print(f"  [{i+1}] Page {doc.metadata.get('page')} | {doc.page_content[:160]}...")
        print()


# Run comparison
compare_retrievers("reinvestigation by specialized agency under Section 482")